In [1]:
import re
import string
import numpy as np
import pandas as pd
import nltk
import mlflow
import mlflow.sklearn
import dagshub

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

nltk.download("stopwords")
nltk.download("wordnet")

dagshub.init(
    repo_owner="Cheeku29",
    repo_name="emotion-detection-mlops",
    mlflow=True
)

print("Tracking URI:", mlflow.get_tracking_uri())

mlflow.set_experiment("Logistic Regression Hyperparameter Tuning v2")

c:\Users\gupta\OneDrive\Desktop\MLOPS\emotion-detection-mlops\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\gupta\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\gupta\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


Accessing as Cheeku29

Initialized MLflow to track repo "Cheeku29/emotion-detection-mlops"

Repository Cheeku29/emotion-detection-mlops initialized!

Tracking URI: https://dagshub.com/Cheeku29/emotion-detection-mlops.mlflow


2026/10/26 03:59:30 INFO mlflow.tracking.fluent: Experiment with name 'Logistic Regression Hyperparameter Tuning v2' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/1a37fdda278f4595a9e83252a2b58227', creation_time=1792967371109, effective_trace_archival_retention=None, experiment_id='3', last_update_time=1792967371109, lifecycle_stage='active', name='Logistic Regression Hyperparameter Tuning v2', tags={}, trace_location=None, workspace='default'>

In [2]:
df = pd.read_csv(
    "https://raw.githubusercontent.com/campusx-official/jupyter-masterclass/main/tweet_emotions.csv"
).drop(columns=["tweet_id"])


stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()


def normalize_text(text):
    text = str(text).lower()
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)
    text = re.sub(r"\d+", " ", text)
    text = text.translate(
        str.maketrans("", "", string.punctuation)
    )
    text = re.sub(r"\s+", " ", text).strip()

    words = [
        lemmatizer.lemmatize(word)
        for word in text.split()
        if word not in stop_words
    ]

    return " ".join(words)


df["content"] = df["content"].apply(normalize_text)

# Keep only happiness and sadness
df = df[
    df["sentiment"].isin(["happiness", "sadness"])
].copy()

df["sentiment"] = df["sentiment"].map({
    "sadness": 0,
    "happiness": 1
})

df = df.dropna(subset=["content", "sentiment"])

X = df["content"]
y = df["sentiment"].astype(int)

print("Dataset shape:", df.shape)
print(y.value_counts())

Dataset shape: (10374, 2)
sentiment
1    5209
0    5165
Name: count, dtype: int64


In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

Training samples: 8299
Testing samples: 2075


In [4]:
pipeline = Pipeline([
    ("bow", CountVectorizer(max_features=3000)),
    ("lr", LogisticRegression(max_iter=1000))
])

param_grid = {
    "lr__C": [0.01, 0.1, 1, 10, 100],
    "lr__penalty": ["l1", "l2"],
    "lr__solver": ["liblinear"]
}

In [5]:
results = []

with mlflow.start_run(
    run_name="LR BoW - Grid Search"
) as parent_run:

    mlflow.log_params({
        "vectorizer": "CountVectorizer",
        "max_features": 3000,
        "cv_folds": 5,
        "scoring": "f1",
        "test_size": 0.2,
        "random_state": 42
    })

    grid_search = GridSearchCV(
        estimator=pipeline,
        param_grid=param_grid,
        cv=5,
        scoring="f1",
        n_jobs=-1,
        verbose=1,
        return_train_score=True
    )

    # Tune only on training data
    grid_search.fit(X_train, y_train)

    # Evaluate the best model once on the held-out test set
    best_model = grid_search.best_estimator_
    y_pred = best_model.predict(X_test)

    test_metrics = {
        "test_accuracy": accuracy_score(y_test, y_pred),
        "test_precision": precision_score(
            y_test, y_pred, zero_division=0
        ),
        "test_recall": recall_score(
            y_test, y_pred, zero_division=0
        ),
        "test_f1_score": f1_score(
            y_test, y_pred, zero_division=0
        )
    }

    mlflow.log_params({
        f"best_{key}": value
        for key, value in grid_search.best_params_.items()
    })

    mlflow.log_metric(
        "best_mean_cv_f1",
        grid_search.best_score_
    )

    mlflow.log_metrics(test_metrics)

    # Log every hyperparameter combination as a child run
    cv_results = grid_search.cv_results_

    for i, params in enumerate(cv_results["params"]):
        with mlflow.start_run(
            run_name=f"LR BoW - Trial {i + 1}",
            nested=True
        ):
            # Remove pipeline prefixes for readable MLflow params
            readable_params = {
                key.replace("lr__", ""): value
                for key, value in params.items()
            }

            mlflow.log_params(readable_params)
            mlflow.log_metric(
                "mean_cv_f1",
                cv_results["mean_test_score"][i]
            )
            mlflow.log_metric(
                "std_cv_f1",
                cv_results["std_test_score"][i]
            )
            mlflow.log_metric(
                "mean_train_f1",
                cv_results["mean_train_score"][i]
            )

            results.append({
                "C": params["lr__C"],
                "Penalty": params["lr__penalty"],
                "CV F1": cv_results["mean_test_score"][i],
                "CV Std": cv_results["std_test_score"][i]
            })

    # Log the complete pipeline, including its fitted vectorizer
    mlflow.sklearn.log_model(
        best_model,
        name="best_bow_logistic_regression",
        serialization_format="pickle"
    )

    print("Best parameters:", grid_search.best_params_)
    print("Best mean CV F1:", grid_search.best_score_)
    print("Test metrics:", test_metrics)

Fitting 5 folds for each of 10 candidates, totalling 50 fits
🏃 View run LR BoW - Trial 1 at: https://dagshub.com/Cheeku29/emotion-detection-mlops.mlflow/#/experiments/3/runs/18a4aac44fbb461ea36f09645693a892
🧪 View experiment at: https://dagshub.com/Cheeku29/emotion-detection-mlops.mlflow/#/experiments/3
🏃 View run LR BoW - Trial 2 at: https://dagshub.com/Cheeku29/emotion-detection-mlops.mlflow/#/experiments/3/runs/e65f05b3a6464dbe960673ec172dabb3
🧪 View experiment at: https://dagshub.com/Cheeku29/emotion-detection-mlops.mlflow/#/experiments/3
🏃 View run LR BoW - Trial 3 at: https://dagshub.com/Cheeku29/emotion-detection-mlops.mlflow/#/experiments/3/runs/620065246bbb41d78080e1408d718cfe
🧪 View experiment at: https://dagshub.com/Cheeku29/emotion-detection-mlops.mlflow/#/experiments/3
🏃 View run LR BoW - Trial 4 at: https://dagshub.com/Cheeku29/emotion-detection-mlops.mlflow/#/experiments/3/runs/ad35503b339e4d5cb64208f2854c3d23
🧪 View experiment at: https://dagshub.com/Cheeku29/emotion-de

2026/10/26 04:02:36 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/10/26 04:02:51 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


Best parameters: {'lr__C': 1, 'lr__penalty': 'l1', 'lr__solver': 'liblinear'}
Best mean CV F1: 0.7897353134280122
Test metrics: {'test_accuracy': 0.7783132530120482, 'test_precision': 0.7664835164835165, 'test_recall': 0.8032629558541267, 'test_f1_score': 0.7844423617619494}
🏃 View run LR BoW - Grid Search at: https://dagshub.com/Cheeku29/emotion-detection-mlops.mlflow/#/experiments/3/runs/d1c13b102a9440ab811bc2a66f9d4182
🧪 View experiment at: https://dagshub.com/Cheeku29/emotion-detection-mlops.mlflow/#/experiments/3


In [6]:
results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    by="CV F1",
    ascending=False
).reset_index(drop=True)

display(results_df)

,C,Penalty,CV F1,CV Std
0,1.00,l1,0.789735,0.010891
1,1.00,l2,0.789046,0.008920
2,0.10,l2,0.786448,0.008154
3,10.00,l2,0.768931,0.009033
4,10.00,l1,0.758610,0.009941
5,0.01,l2,0.753264,0.010325
6,100.00,l2,0.743472,0.008122
7,100.00,l1,0.719332,0.008796
8,0.10,l1,0.701299,0.026342
9,0.01,l1,0.288321,0.027294
